In [ ]:
{
 "nbformat": 4,
 "nbformat_minor": 5,
 "metadata": {
  "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
  "language_info": {"name": "python", "version": "3.10.0"}
 },
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["# 🐔 Poultry AI — Model Training & Evaluation\n", "Notebook 03 | Projects 1 & 2: Profit Prediction + Risk Classification"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys; sys.path.insert(0, '..')\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from sklearn.model_selection import train_test_split, cross_val_score\n",
    "from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,\n",
    "                              classification_report, confusion_matrix, ConfusionMatrixDisplay)\n",
    "from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier\n",
    "from sklearn.preprocessing import LabelEncoder\n",
    "\n",
    "from configs.config import TEST_SIZE, RANDOM_STATE, TARGET_PROFIT, TARGET_RISK\n",
    "from src.preprocessing.data_loader import load_raw, clean, get_X_y\n",
    "\n",
    "sns.set_theme(style='whitegrid')\n",
    "df = clean(load_raw())\n",
    "print(f'Dataset: {df.shape}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Project 1 — Profit Prediction"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "X, y = get_X_y(df, TARGET_PROFIT)\n",
    "X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)\n",
    "\n",
    "rf_reg = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1)\n",
    "rf_reg.fit(X_tr, y_tr)\n",
    "y_pred = rf_reg.predict(X_te)\n",
    "\n",
    "rmse = np.sqrt(mean_squared_error(y_te, y_pred))\n",
    "mae  = mean_absolute_error(y_te, y_pred)\n",
    "r2   = r2_score(y_te, y_pred)\n",
    "print(f'RMSE : ₹{rmse:,.0f}')\n",
    "print(f'MAE  : ₹{mae:,.0f}')\n",
    "print(f'R²   : {r2:.4f}')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "# Actual vs Predicted scatter\n",
    "axes[0].scatter(y_te, y_pred, alpha=0.35, s=12, color='#2563EB')\n",
    "mn, mx = y_te.min(), y_te.max()\n",
    "axes[0].plot([mn, mx], [mn, mx], 'r--', lw=2)\n",
    "axes[0].set_xlabel('Actual Profit (₹)')\n",
    "axes[0].set_ylabel('Predicted Profit (₹)')\n",
    "axes[0].set_title(f'Actual vs Predicted  (R²={r2:.3f})')\n",
    "\n",
    "# Residuals\n",
    "residuals = y_te - y_pred\n",
    "axes[1].hist(residuals, bins=40, color='#0f766e', edgecolor='white')\n",
    "axes[1].axvline(0, color='red', linestyle='--')\n",
    "axes[1].set_title('Residual Distribution')\n",
    "axes[1].set_xlabel('Residual (₹)')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Project 2 — Risk Classification"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "X, y_raw = get_X_y(df, TARGET_RISK)\n",
    "le = LabelEncoder()\n",
    "y  = le.fit_transform(y_raw)\n",
    "\n",
    "X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE,\n",
    "                                            stratify=y, random_state=RANDOM_STATE)\n",
    "rf_clf = RandomForestClassifier(n_estimators=200, max_depth=12,\n",
    "                                class_weight='balanced',\n",
    "                                random_state=RANDOM_STATE, n_jobs=-1)\n",
    "rf_clf.fit(X_tr, y_tr)\n",
    "y_pred_clf = rf_clf.predict(X_te)\n",
    "\n",
    "print(classification_report(y_te, y_pred_clf, target_names=le.classes_))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "# Confusion matrix\n",
    "cm = confusion_matrix(y_te, y_pred_clf)\n",
    "sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',\n",
    "            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[0])\n",
    "axes[0].set_title('Confusion Matrix')\n",
    "axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')\n",
    "\n",
    "# Feature importances\n",
    "fi = pd.Series(rf_clf.feature_importances_, index=X.columns).sort_values(ascending=True).tail(10)\n",
    "fi.plot(kind='barh', ax=axes[1], color='#7c3aed')\n",
    "axes[1].set_title('Top 10 Feature Importances (Risk Classifier)')\n",
    "axes[1].set_xlabel('Importance')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Cross-Validation Summary"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "cv_r2  = cross_val_score(rf_reg, X, get_X_y(df, TARGET_PROFIT)[1], cv=5, scoring='r2')\n",
    "cv_acc = cross_val_score(rf_clf, X, y, cv=5, scoring='accuracy')\n",
    "\n",
    "summary = pd.DataFrame({\n",
    "    'Model': ['Profit Regressor (RF)', 'Risk Classifier (RF)'],\n",
    "    'CV Mean': [f'{cv_r2.mean():.4f} R²', f'{cv_acc.mean():.4f} Acc'],\n",
    "    'CV Std':  [f'±{cv_r2.std():.4f}',   f'±{cv_acc.std():.4f}'],\n",
    "})\n",
    "summary"
   ]
  }
 ]
}
